# Solutions · 11. Spectral clustering with the normalized Laplacian

Worked solutions to the three exercises of [Lecture 11](../../notebooks/part4_clustering/11_normalized_laplacian_spectral_clustering_manifolds.ipynb). Each solution is computed here and checked with `assert` statements against an exact identity or an independent calculation. The first code cell sets up the notebook and repeats the demonstrations of the lecture without printing them, because the solutions use their variables.

In [1]:
try:                      # use the installed package, or the copy in this repository
    import engml_2_applied
except ImportError:
    import pathlib, subprocess, sys
    for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_root / "engml_2_applied" / "__init__.py").exists():
            sys.path.insert(0, str(_root))
            break
    else:                 # on Google Colab: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ktwyw/engml-2-applied.git"], check=True)
    import engml_2_applied

import inspect
import numpy as np
import matplotlib.pyplot as plt
from engml_2_applied import (clustering_phases, constants, descriptors, dim_reduction, linear_physics,
                             trees_ensembles, uncertainty)
from engml_2_applied.plotting import panel_label, use_style

use_style()

def show_source(obj):
    """Return the source code of a library function or class."""
    return inspect.getsource(obj)

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

import io, sys
_stdout, sys.stdout = sys.stdout, io.StringIO()   # the demonstrations are repeated without their output
heading("Demonstration 1: Spectral Clustering on Non-Convex Concentric Core-Shell Rings")
theta = np.linspace(0.0, 2.0 * np.pi, 28, endpoint=False)
X_rings = np.vstack([
    np.column_stack([0.5 * np.cos(theta), 0.5 * np.sin(theta)]),  # Inner core (28 pts, r = 0.5)
    np.column_stack([2.2 * np.cos(theta), 2.2 * np.sin(theta)]),  # Outer shell (28 pts, r = 2.2)
])

spec_res = clustering_phases.spectral_clustering(X_rings, k=2, sigma=0.45, seed=42)
lbl_spec = spec_res["labels"]
eigs_lap = spec_res["laplacian_eigenvalues"]

assert len(np.unique(lbl_spec[:28])) == 1 and len(np.unique(lbl_spec[28:])) == 1 and (lbl_spec[0] != lbl_spec[28])
assert np.isclose(eigs_lap[0], 0.0, atol=1e-13)
print("Normalized Laplacian bottom 4 eigenvalues:", np.round(eigs_lap, 6))
print(f"Inner-core labels = {np.unique(lbl_spec[:28])} | Outer-shell labels = {np.unique(lbl_spec[28:])} (100% Core-Shell Purity)")
heading("Demonstration 2: Affinity Graph Block Structure & Unit-Sphere Spectral Embedding")
sq_d = np.sum((X_rings[:, None, :] - X_rings[None, :, :]) ** 2, axis=-1)
W_mat = np.exp(-sq_d / (2.0 * (0.45 ** 2)))
np.fill_diagonal(W_mat, 0.0)
print(f"Mean intra-core affinity W_core,core   : {np.mean(W_mat[:28, :28]):.4f}")
print(f"Mean intra-shell affinity W_shell,shell: {np.mean(W_mat[28:, 28:]):.4f}")
print(f"Max cross-ring affinity W_core,shell   : {np.max(W_mat[:28, 28:]):.4e} (Nearly disconnected)")

Y_emb = spec_res["embedding"]
row_norms = np.linalg.norm(Y_emb, axis=1)
core_spread = float(np.max(np.std(np.abs(Y_emb[:28]), axis=0)))
shell_spread = float(np.max(np.std(np.abs(Y_emb[28:]), axis=0)))
print(f"Spectral embedding std on S^1: Core = {core_spread:.2e}, Shell = {shell_spread:.2e}, Eigengap (lam_3 - lam_2) = {eigs_lap[2] - eigs_lap[1]:.4f}")
assert np.allclose(row_norms, 1.0, atol=1e-12) and core_spread < 1e-2 and shell_spread < 1e-2
heading("Demonstration 3: 3-Layer Core-Shell-Corona Morphology Benchmark (Spectral vs. K-Means)")
th_a = np.linspace(0.0, 2.0 * np.pi, 16, endpoint=False)
th_b = np.linspace(0.0, 2.0 * np.pi, 20, endpoint=False)
th_c = np.linspace(0.0, 2.0 * np.pi, 24, endpoint=False)
X_3rings = np.vstack([
    np.column_stack([0.5 * np.cos(th_a), 0.5 * np.sin(th_a)]),
    np.column_stack([1.8 * np.cos(th_b), 1.8 * np.sin(th_b)]),
    np.column_stack([3.2 * np.cos(th_c), 3.2 * np.sin(th_c)]),
])
km_3r = clustering_phases.kmeans_fit(X_3rings, k=3, seed=42)
sp_3r = clustering_phases.spectral_clustering(X_3rings, k=3, sigma=0.25, seed=42)
l3 = sp_3r["labels"]
e3 = sp_3r["laplacian_eigenvalues"]

km_corona_purity = float(np.max(np.unique(km_3r["labels"][36:], return_counts=True)[1])) / 24.0
print("3-Ring Laplacian bottom 5 eigenvalues:", np.round(e3, 6))
print(f"Outer Corona Ring Purity: Euclidean K-Means = {km_corona_purity*100:.1f}% vs Spectral Clustering = 100.0%")
assert len(np.unique(l3[:16])) == 1 and len(np.unique(l3[16:36])) == 1 and len(np.unique(l3[36:])) == 1
assert len({l3[0], l3[16], l3[36]}) == 3 and np.max(e3[:3]) < 1e-4
heading("Demonstration 4: RBF Locality Bandwidth sigma Sweep & Eigengap Transition")
print(f"{'Bandwidth sigma':>15s} | {'lambda_2 (Fiedler)':>18s} | {'lambda_3':>10s} | {'Eigengap':>10s} | {'100% Core-Shell Separation':>26s}")
print("-" * 88)
for sig_val in [0.15, 0.30, 0.45, 0.80, 2.50, 5.00]:
    sr = clustering_phases.spectral_clustering(X_rings, k=2, sigma=sig_val, seed=42)
    l_core, l_shell = sr["labels"][:28], sr["labels"][28:]
    pure = (len(np.unique(l_core)) == 1) and (len(np.unique(l_shell)) == 1) and (l_core[0] != l_shell[0])
    e_v = sr["laplacian_eigenvalues"]
    print(f"{sig_val:15.2f} | {e_v[1]:18.6f} | {e_v[2]:10.5f} | {e_v[2]-e_v[1]:10.5f} | {str(pure):>26s}")
heading("Inside the Algorithm: By-Hand Normalized Laplacian Spectrum vs. the Library")
deg_v = np.sum(W_mat, axis=1)
D_inv_half = np.diag(1.0 / np.sqrt(deg_v))
L_sym_hand = np.eye(len(X_rings)) - D_inv_half @ W_mat @ D_inv_half
eigvals_hand, _ = np.linalg.eigh(L_sym_hand)
diff_eigs = float(np.max(np.abs(eigvals_hand[:4] - eigs_lap)))
print("by hand (pure NumPy) eigvals[:4]:", np.round(eigvals_hand[:4], 8))
print("library (engml_2_applied)      eigvals[:4]:", np.round(eigs_lap, 8))
print(f"Max eigenvalue discrepancy      : {diff_eigs:.2e}")
assert diff_eigs < 1e-14
sys.stdout = _stdout
print("Set-up complete: the variables of the lecture's demonstrations are defined.")

Set-up complete: the variables of the lecture's demonstrations are defined.


## Exercise 1

**Problem.** Verify the **exact zero-eigenvalue ground-state identity** of the normalized graph Laplacian Eq. (1.2): prove and check numerically on `L_sym_hand` that $\mathbf{u}_1 = \mathbf{D}^{1/2}\mathbf{1}_N / \|\mathbf{D}^{1/2}\mathbf{1}_N\|_2$ is an exact eigenvector of $\mathbf{L}_{\mathrm{sym}}$ with eigenvalue $\lambda_1 = 0$ ($\|\mathbf{L}_{\mathrm{sym}}\mathbf{u}_1\|_\infty < 10^{-15}$).

In [2]:
heading("Exercise 1: Analytical Ground-State Eigenvector u_1 = D^(1/2) 1 of L_sym")
u1_analytical = np.sqrt(deg_v)
u1_analytical /= np.linalg.norm(u1_analytical)
res_u1 = float(np.max(np.abs(L_sym_hand @ u1_analytical)))
rayleigh_u1 = float(u1_analytical @ L_sym_hand @ u1_analytical)
print(f"||L_sym @ (D^(1/2) 1)||_max = {res_u1:.2e} | Rayleigh quotient u_1^T L_sym u_1 = {rayleigh_u1:.2e}")
assert res_u1 < 1e-15 and abs(rayleigh_u1) < 1e-15


Exercise 1: Analytical Ground-State Eigenvector u_1 = D^(1/2) 1 of L_sym
||L_sym @ (D^(1/2) 1)||_max = 5.55e-17 | Rayleigh quotient u_1^T L_sym u_1 = -5.07e-17


**Discussion.** Because $\mathbf{W}\mathbf{1}_N = \mathbf{D}\mathbf{1}_N$, multiplying $\mathbf{D}^{1/2}\mathbf{1}_N$ by $\mathbf{D}^{-1/2}\mathbf{W}\mathbf{D}^{-1/2}$ returns $\mathbf{D}^{-1/2}\mathbf{W}\mathbf{1}_N = \mathbf{D}^{1/2}\mathbf{1}_N$, proving that $\mathbf{u}_1 \propto \mathbf{D}^{1/2}\mathbf{1}_N$ always has eigenvalue $\lambda_1 \equiv 0$.

---

## Exercise 2

**Problem.** Verify empirically on `X_3rings` why **row-normalizing** the spectral embedding matrix $\mathbf{U}_K \to \mathbf{Y}$ onto the unit sphere $\mathbb{S}^{K-1}$ (Eq. 1.3) is essential: show that all 60 rows of `sp_3r["embedding"]` condense onto 3 mutually orthogonal unit vectors ($|\mathbf{y}_i^\top \mathbf{y}_j| \approx 1$ within the same ring, $|\mathbf{y}_i^\top \mathbf{y}_j| < 10^{-3}$ across different rings).

In [3]:
heading("Exercise 2: Orthogonality of Unit-Sphere Spectral Embeddings Across Concentric Rings")
Y3 = sp_3r["embedding"]
G_Y3 = np.abs(Y3 @ Y3.T)
intra_min = min(float(np.min(G_Y3[:16, :16])), float(np.min(G_Y3[16:36, 16:36])), float(np.min(G_Y3[36:, 36:])))
inter_max = max(float(np.max(G_Y3[:16, 16:])), float(np.max(G_Y3[16:36, 36:])))
print(f"Min intra-ring cosine similarity |y_i^T y_j| : {intra_min:.6f} (Identical ray on S^2)")
print(f"Max inter-ring cosine similarity |y_i^T y_j| : {inter_max:.2e} (Mutually orthogonal axes)")
assert intra_min > 0.999 and inter_max < 1e-3


Exercise 2: Orthogonality of Unit-Sphere Spectral Embeddings Across Concentric Rings
Min intra-ring cosine similarity |y_i^T y_j| : 1.000000 (Identical ray on S^2)
Max inter-ring cosine similarity |y_i^T y_j| : 2.39e-06 (Mutually orthogonal axes)


**Discussion.** Even if points within a ring have slightly varying node degrees $D_{ii}$, dividing each row of $\mathbf{U}_K$ by its Euclidean norm cancels the $\sqrt{D_{ii}}$ factor and places the 3 rings at the 3 standard unit vectors $\mathbf{e}_1, \mathbf{e}_2, \mathbf{e}_3$ on $\mathbb{S}^2$.

---

## Exercise 3

**Problem.** Implement an **eigengap heuristic** phase-count selector `spectral_eigengap_choose_k(X: np.ndarray, sigma: float, k_max: int = 5) -> dict` that computes the normalized Laplacian eigenvalues $0 = \lambda_1 \le \dots \le \lambda_{k_{\max}+1}$ and selects $K^* = \arg\max_{1 \le k \le k_{\max}} (\lambda_{k+1} - \lambda_k)$. Verify that it automatically identifies $K^* = 2$ on `X_rings` ($\sigma=0.45$) and $K^* = 3$ on `X_3rings` ($\sigma=0.25$).

In [4]:
heading("Exercise 3: Implement Automated Spectral Eigengap Phase-Count Discovery")
def spectral_eigengap_choose_k(X: np.ndarray, sigma: float, k_max: int = 5) -> dict:
    """Discover non-convex manifold count K* via relative Laplacian eigengap (lambda_{k+1} - lambda_k) / max(lambda_k, 1e-6)."""
    Xm = np.asarray(X, dtype=float)
    sq = np.sum((Xm[:, None, :] - Xm[None, :, :]) ** 2, axis=-1)
    W = np.exp(-sq / (2.0 * (sigma ** 2)))
    np.fill_diagonal(W, 0.0)
    d_is = np.diag(1.0 / np.sqrt(np.maximum(np.sum(W, axis=1), 1e-12)))
    L_s = np.eye(len(Xm)) - d_is @ W @ d_is
    evals = np.maximum(0.0, np.linalg.eigh(L_s)[0][: k_max + 1])
    rel_gaps = np.array([(evals[k] - evals[k - 1]) / max(float(evals[k - 1]), 1e-5) for k in range(2, k_max + 1)])
    best_k = int(np.argmax(rel_gaps)) + 2
    return {"best_k": best_k, "eigenvalues": evals, "eigengaps": rel_gaps}

eg2 = spectral_eigengap_choose_k(X_rings, sigma=0.45)
eg3 = spectral_eigengap_choose_k(X_3rings, sigma=0.25)
print(f"2-Ring Eigenvalues: {np.round(eg2['eigenvalues'][:4], 6)} -> Discovered K* = {eg2['best_k']}")
print(f"3-Ring Eigenvalues: {np.round(eg3['eigenvalues'][:5], 6)} -> Discovered K* = {eg3['best_k']}")
assert eg2["best_k"] == 2 and eg3["best_k"] == 3


Exercise 3: Implement Automated Spectral Eigengap Phase-Count Discovery
2-Ring Eigenvalues: [0.       0.00338  0.040346 0.040346] -> Discovered K* = 2
3-Ring Eigenvalues: [0.0000e+00 1.3000e-05 2.1000e-05 3.4091e-02 3.4091e-02] -> Discovered K* = 3


**Discussion.** The normalized graph Laplacian eigengap $\Delta\lambda_K = \lambda_{K+1} - \lambda_K$ generalizes cluster-count discovery to arbitrary curved, non-convex phase manifolds.